# QRT Data Challenge

## Project objective

The task is to predict whether each allocation's future return is positive. The leaderboard metric is **classification accuracy**.

This notebook is the presentation version of the original project: it keeps the benchmark, feature research, model comparison, purged validation, threshold calibration, ensemble check, final CatBoost model, and submission generation, while removing exploratory and duplicated code.

## 1. Imports and configuration

**What:** load the small set of libraries used by the final workflow and fix the random seed.  
**Why:** every result should be reproducible from a fresh kernel.  
**Decision:** use relative paths and the same model seeds and parameters as the original work.

In [1]:
from pathlib import Path
import gc
import itertools
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

import sklearn
import lightgbm as lgb
import xgboost
import catboost
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = Path.cwd()
RANDOM_STATE = 42
N_JOBS = -1
TARGET_ENCODING_SMOOTHING = 20.0

np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)
warnings.filterwarnings("ignore", category=FutureWarning)

print({
    "python_stack": f"pandas {pd.__version__}, scikit-learn {sklearn.__version__}",
    "boosting": f"LightGBM {lgb.__version__}, XGBoost {xgboost.__version__}, CatBoost {catboost.__version__}",
    "random_seed": RANDOM_STATE,
})

{'python_stack': 'pandas 2.3.3, scikit-learn 1.7.2', 'boosting': 'LightGBM 4.7.0, XGBoost 3.2.0, CatBoost 1.2.10', 'random_seed': 42}


## 2. Data loading and quick exploration

**What:** locate the four challenge files, validate IDs and schemas, and show only compact aggregates.  
**Why:** the hashed filenames may change and the private rows should not be printed.  
**Observation:** train and test have the same predictor schema and the target is close to balanced.  
**Decision:** keep the data local and load it from either the repository root or an ignored `data/` directory.

In [2]:
def locate_data_file(pattern):
    """Return the unique matching file from the project root or data/."""
    matches = []
    for folder in (PROJECT_DIR, PROJECT_DIR / "data"):
        if folder.exists():
            matches.extend(folder.glob(pattern))
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one file matching {pattern!r}; found {len(matches)}: "
            f"{[path.name for path in matches]}"
        )
    return matches[0]

DATA_PATHS = {
    "X_train": locate_data_file("X_train_*.csv"),
    "X_test": locate_data_file("X_test_*.csv"),
    "y_train": locate_data_file("y_train_*.csv"),
    "sample_submission": locate_data_file("sample_submission_*.csv"),
}

X_train = pd.read_csv(DATA_PATHS["X_train"])
X_test = pd.read_csv(DATA_PATHS["X_test"])
y_train_file = pd.read_csv(DATA_PATHS["y_train"])
sample_submission = pd.read_csv(DATA_PATHS["sample_submission"])

assert X_train["ROW_ID"].is_unique and X_test["ROW_ID"].is_unique
assert y_train_file["ROW_ID"].is_unique and sample_submission["ROW_ID"].is_unique
assert X_train["ROW_ID"].equals(y_train_file["ROW_ID"])
assert X_test["ROW_ID"].equals(sample_submission["ROW_ID"])
assert list(X_train.columns) == list(X_test.columns)
assert list(sample_submission.columns) == ["ROW_ID", "prediction"]

y_continuous = y_train_file["target"].to_numpy(dtype=float)
y_binary = (y_continuous > 0).astype(np.int8)

print("Loaded:", {name: path.relative_to(PROJECT_DIR).as_posix() for name, path in DATA_PATHS.items()})

Loaded: {'X_train': 'X_train_9xQjqvZ.csv', 'X_test': 'X_test_1zTtEnD.csv', 'y_train': 'y_train_Ppwhaz8.csv', 'sample_submission': 'sample_submission_SpGVFuH.csv'}


In [3]:
dataset_overview = pd.DataFrame([
    {
        "split": "train", "rows": len(X_train), "predictors": X_train.shape[1],
        "timestamps": X_train["TS"].nunique(), "allocations": X_train["ALLOCATION"].nunique(),
        "groups": X_train["GROUP"].nunique(),
    },
    {
        "split": "test", "rows": len(X_test), "predictors": X_test.shape[1],
        "timestamps": X_test["TS"].nunique(), "allocations": X_test["ALLOCATION"].nunique(),
        "groups": X_test["GROUP"].nunique(),
    },
])

missingness = (
    X_train.isna().mean().mul(100).sort_values(ascending=False).head(6)
    .rename("missing_percent").round(3).to_frame()
)

target_overview = pd.DataFrame([{
    "negative_rows": int((y_binary == 0).sum()),
    "positive_rows": int((y_binary == 1).sum()),
    "positive_rate": float(y_binary.mean()),
    "zero_targets": int((y_continuous == 0).sum()),
}])

assert X_train.groupby("ALLOCATION")["GROUP"].nunique().max() == 1
assert set(X_test["ALLOCATION"]).issubset(set(X_train["ALLOCATION"]))

display(dataset_overview, target_overview.round(6), missingness)

,split,rows,predictors,timestamps,allocations,groups
0,train,527073,45,2522,278,4
1,test,31870,45,120,278,4


,negative_rows,positive_rows,positive_rate,zero_targets
0,259750,267323,0.507184,0


,missing_percent
SIGNED_VOLUME_1,73.520
SIGNED_VOLUME_20,1.612
MEDIAN_DAILY_TURNOVER,0.690
SIGNED_VOLUME_19,0.570
SIGNED_VOLUME_18,0.163
SIGNED_VOLUME_17,0.011


## 3. Prediction target

The supplied target is a continuous future return, but the challenge scores only its sign. The modelling label is therefore:

`binary_target = 1 if target > 0 else 0`

The target is nearly balanced, so accuracy is appropriate and no class weighting is required. Regression remains useful for reproducing the official benchmark, while the final model directly optimizes binary classification.

## 4. Validation strategy

**What:** keep every timestamp intact and compare two validation views.  
**Why:** rows at the same timestamp share cross-sectional context, while random row splits would leak that structure.  
**Observation:** shuffled timestamp folds reproduce the official benchmark; chronological expanding folds give a more realistic final check.  
**Decision:** use eight shuffled timestamp folds only for historical comparability, and select the final model on four expanding folds separated from validation by a 20-timestamp embargo.

In [4]:
def ordered_timestamps(values):
    """Order labels such as DATE_0001 by their numeric suffix."""
    values = pd.Series(pd.unique(pd.Series(values).astype(str)))
    suffix = pd.to_numeric(values.str.extract(r"(\d+)$", expand=False), errors="coerce")
    order = pd.DataFrame({"TS": values, "suffix": suffix}).sort_values(
        ["suffix", "TS"], kind="stable"
    )
    return order["TS"].to_numpy()

def rows_for_timestamps(frame, timestamps):
    return np.flatnonzero(frame["TS"].astype(str).isin(set(timestamps)).to_numpy())

def make_shuffled_timestamp_folds(frame, n_splits=8, random_state=0):
    timestamps = pd.unique(frame["TS"].astype(str))
    folds = []
    for fold, (train_pos, valid_pos) in enumerate(
        KFold(n_splits=n_splits, shuffle=True, random_state=random_state).split(timestamps)
    ):
        folds.append({
            "fold": fold,
            "train_idx": rows_for_timestamps(frame, timestamps[train_pos]),
            "valid_idx": rows_for_timestamps(frame, timestamps[valid_pos]),
        })
    return folds

def make_purged_folds(frame, n_splits=4, train_fraction=0.5, valid_fraction=0.1, embargo=20):
    timestamps = ordered_timestamps(frame["TS"])
    minimum_train = int(np.ceil(len(timestamps) * train_fraction))
    valid_size = int(np.floor(len(timestamps) * valid_fraction))
    starts = np.linspace(minimum_train + embargo, len(timestamps) - valid_size, n_splits, dtype=int)
    folds = []
    for fold, valid_start in enumerate(starts):
        train_end = valid_start - embargo
        train_ts = timestamps[:train_end]
        embargo_ts = timestamps[train_end:valid_start]
        valid_ts = timestamps[valid_start:valid_start + valid_size]
        train_idx = rows_for_timestamps(frame, train_ts)
        valid_idx = rows_for_timestamps(frame, valid_ts)
        assert set(train_ts).isdisjoint(valid_ts)
        assert len(embargo_ts) == embargo
        folds.append({
            "fold": fold, "train_idx": train_idx, "valid_idx": valid_idx,
            "train_ts": train_ts, "embargo_ts": embargo_ts, "valid_ts": valid_ts,
        })
    return folds

official_folds = make_shuffled_timestamp_folds(X_train)
purged_folds = make_purged_folds(X_train)

validation_table = pd.DataFrame([{
    "fold": fold["fold"],
    "train_period": f"{fold['train_ts'][0]}–{fold['train_ts'][-1]}",
    "embargo_period": f"{fold['embargo_ts'][0]}–{fold['embargo_ts'][-1]}",
    "validation_period": f"{fold['valid_ts'][0]}–{fold['valid_ts'][-1]}",
    "train_rows": len(fold["train_idx"]),
    "validation_rows": len(fold["valid_idx"]),
} for fold in purged_folds])

display(validation_table)

,fold,train_period,embargo_period,validation_period,train_rows,validation_rows
0,0,DATE_0001–DATE_1261,DATE_1262–DATE_1281,DATE_1282–DATE_1533,263504,54491
1,1,DATE_0001–DATE_1590,DATE_1591–DATE_1610,DATE_1611–DATE_1862,335074,50936
2,2,DATE_0001–DATE_1920,DATE_1921–DATE_1940,DATE_1941–DATE_2192,401669,52504
3,3,DATE_0001–DATE_2250,DATE_2251–DATE_2270,DATE_2271–DATE_2522,469981,52826


The embargo mirrors the longest 20-lag input window and reduces adjacency between training and validation. It is a conservative design choice; the challenge material does not provide enough information to interpret it as a formal proof of label-horizon independence.

## 5. Official benchmark

**What:** reproduce the challenge's LightGBM regression benchmark and compare it with the simple `RET_1` sign rule.  
**Why:** this provides a known reference before adding features or changing objectives.  
**Observation:** the official model improves on recent-return momentum, but only modestly.  
**Decision:** retain the benchmark feature family, then test controlled additions.

In [5]:
RET_COLUMNS = [f"RET_{lag}" for lag in range(1, 21)]
VOLUME_COLUMNS = [f"SIGNED_VOLUME_{lag}" for lag in range(1, 21)]

def build_benchmark_features(frame):
    """Reproduce the 53 predictors supplied in the official benchmark."""
    features = frame[RET_COLUMNS + VOLUME_COLUMNS + ["MEDIAN_DAILY_TURNOVER"]].copy()
    for horizon in (3, 5, 10, 15, 20):
        average = frame[RET_COLUMNS[:horizon]].mean(axis=1)
        features[f"AVERAGE_PERF_{horizon}"] = average
        features[f"ALLOCATIONS_AVERAGE_PERF_{horizon}"] = average.groupby(frame["TS"]).transform("mean")
    volatility = frame[RET_COLUMNS].std(axis=1)
    features["STD_PERF_20"] = volatility
    features["ALLOCATIONS_STD_PERF_20"] = volatility.groupby(frame["TS"]).transform("mean")
    assert features.shape[1] == 53
    return features.replace([np.inf, -np.inf], np.nan)

def run_official_benchmark(features, target, folds):
    oof = np.full(len(features), np.nan)
    records = []
    parameters = {
        "objective": "mse", "metric": "mse", "num_threads": 50,
        "seed": RANDOM_STATE, "verbosity": -1, "learning_rate": 0.01, "max_depth": 3,
    }
    started = time.perf_counter()
    for fold in folds:
        train_idx, valid_idx = fold["train_idx"], fold["valid_idx"]
        training_data = lgb.Dataset(features.iloc[train_idx], label=target[train_idx])
        model = lgb.train(parameters, training_data, num_boost_round=500)
        oof[valid_idx] = model.predict(features.iloc[valid_idx])
        records.append({
            "fold": fold["fold"],
            "accuracy": accuracy_score(target[valid_idx] > 0, oof[valid_idx] > 0),
            "validation_rows": len(valid_idx),
        })
        del training_data, model
    fold_results = pd.DataFrame(records)
    return {
        "fold_results": fold_results,
        "mean_accuracy": fold_results["accuracy"].mean(),
        "global_oof_accuracy": accuracy_score(target > 0, oof > 0),
        "runtime_seconds": time.perf_counter() - started,
    }

benchmark_features = build_benchmark_features(X_train)
official_benchmark = run_official_benchmark(benchmark_features, y_continuous, official_folds)
ret1_accuracy = accuracy_score(y_binary, X_train["RET_1"].to_numpy() > 0)

benchmark_summary = pd.DataFrame([
    {"model": "RET_1 sign rule", "validation": "all rows (deterministic rule)", "accuracy": ret1_accuracy},
    {"model": "Official LightGBM regression", "validation": "8-fold shuffled timestamp CV",
     "accuracy": official_benchmark["mean_accuracy"]},
])
display(benchmark_summary.round({"accuracy": 6}), official_benchmark["fold_results"].round(6))

del benchmark_features
_ = gc.collect()

,model,validation,accuracy
0,RET_1 sign rule,all rows (deterministic rule),0.518924
1,Official LightGBM regression,8-fold shuffled timestamp CV,0.522194


,fold,accuracy,validation_rows
0,0,0.519531,65998
1,1,0.524469,66349
2,2,0.521779,64972
3,3,0.521041,64565
4,4,0.519737,67413
5,5,0.524815,66229
6,6,0.524521,66209
7,7,0.521657,65338


## 6. Feature engineering and preprocessing

**What:** add only the feature groups retained by the executed ablation study.  
**Why:** recent returns contain the strongest simple signal, but their reliability depends on strength, consistency, and cross-sectional position. Allocation history also carries stable information.  
**Observation:** the final representation contains 53 benchmark features, 24 timestamp-relative features, 15 `RET_1` confidence features, and one fold-safe allocation encoding.  
**Decision:** use this same 93-feature pipeline for every final model so the comparison changes the estimator, not the data.

In [6]:
def safe_ratio(numerator, denominator):
    denominator = pd.to_numeric(denominator, errors="coerce")
    return pd.to_numeric(numerator, errors="coerce") / denominator.mask(denominator.abs() <= 1e-12)

def sign_agreement(left, right):
    left, right = pd.to_numeric(left, errors="coerce"), pd.to_numeric(right, errors="coerce")
    observed = left.notna() & right.notna()
    return pd.Series(np.where(observed, (np.sign(left) == np.sign(right)).astype(float), np.nan), index=left.index)

def build_features(frame):
    """Build the retained 92 numeric features without using the target."""
    features = build_benchmark_features(frame)

    bases = {
        "RET1": frame["RET_1"],
        "RET_MEAN3": frame[RET_COLUMNS[:3]].mean(axis=1),
        "RET_MEAN5": frame[RET_COLUMNS[:5]].mean(axis=1),
        "RET_STD5": frame[RET_COLUMNS[:5]].std(axis=1),
        "RET_STD20": frame[RET_COLUMNS].std(axis=1),
        "SIGNED_VOLUME1": frame["SIGNED_VOLUME_1"],
        "SIGNED_VOLUME_MEAN5": frame[VOLUME_COLUMNS[:5]].mean(axis=1),
        "TURNOVER": frame["MEDIAN_DAILY_TURNOVER"],
    }
    cross_sectional = {}
    for name, values in bases.items():
        grouped = values.groupby(frame["TS"])
        timestamp_mean = grouped.transform("mean")
        cross_sectional[f"CS_{name}_TS_PERCENTILE"] = grouped.rank(pct=True, method="average")
        cross_sectional[f"CS_{name}_TS_ZSCORE"] = safe_ratio(values - timestamp_mean, grouped.transform("std"))
        cross_sectional[f"CS_{name}_MINUS_TS_MEAN"] = values - timestamp_mean
    features = pd.concat([features, pd.DataFrame(cross_sectional, index=frame.index)], axis=1)

    ret1 = frame["RET_1"]
    ret2 = frame["RET_2"]
    mean3 = frame[RET_COLUMNS[:3]].mean(axis=1)
    mean5 = frame[RET_COLUMNS[:5]].mean(axis=1)
    mean20 = frame[RET_COLUMNS].mean(axis=1)
    std5 = frame[RET_COLUMNS[:5]].std(axis=1)
    std20 = frame[RET_COLUMNS].std(axis=1)
    mean_abs5 = frame[RET_COLUMNS[:5]].abs().mean(axis=1)
    mean_abs20 = frame[RET_COLUMNS].abs().mean(axis=1)
    confidence = pd.DataFrame({
        "R1C_ABS_RET1": ret1.abs(),
        "R1C_RET1_SQUARED": ret1.pow(2),
        "R1C_SIGN_RET1": np.sign(ret1),
        "R1C_RET1_OVER_STD5": safe_ratio(ret1, std5),
        "R1C_RET1_OVER_STD20": safe_ratio(ret1, std20),
        "R1C_RET1_MINUS_MEAN3": ret1 - mean3,
        "R1C_RET1_MINUS_MEAN5": ret1 - mean5,
        "R1C_RET1_MINUS_MEAN20": ret1 - mean20,
        "R1C_ABS_RET1_MINUS_MEAN5": (ret1 - mean5).abs(),
        "R1C_RET1_OVER_MEAN_ABS5": safe_ratio(ret1, mean_abs5),
        "R1C_RET1_OVER_MEAN_ABS20": safe_ratio(ret1, mean_abs20),
        "R1C_RET1_NEAR_ZERO": np.where(std5.notna(), (ret1.abs() <= 0.25 * std5).astype(float), np.nan),
        "R1C_ABS_RET1_ABOVE_STD5": np.where(std5.notna(), (ret1.abs() > std5).astype(float), np.nan),
        "R1C_SIGN_AGREES_MEAN3": sign_agreement(ret1, mean3),
        "R1C_SIGN_AGREES_RET2": sign_agreement(ret1, ret2),
    }, index=frame.index)
    features = pd.concat([features, confidence], axis=1).replace([np.inf, -np.inf], np.nan)
    assert features.shape[1] == 92 and features.select_dtypes(exclude=np.number).empty
    return features

features_train = build_features(X_train)
feature_inventory = pd.DataFrame({
    "feature_group": ["Official benchmark", "Cross-sectional context", "RET_1 confidence", "Allocation target encoding"],
    "feature_count": [53, 24, 15, 1],
})
display(feature_inventory, pd.DataFrame([{"rows": len(features_train), "numeric_features_before_encoding": features_train.shape[1]}]))

,feature_group,feature_count
0,Official benchmark,53
1,Cross-sectional context,24
2,RET_1 confidence,15
3,Allocation target encoding,1


,rows,numeric_features_before_encoding
0,527073,92


### Fold-safe preprocessing

For each outer fold, medians and the `StandardScaler` are learned on training rows only. The allocation positive-rate encoding uses five inner shuffled timestamp folds, smoothing of 20, and the training-fold global rate as fallback. Validation rows never contribute labels to their encoding. This reproduces the final pipeline used in the original project.

In [7]:
def allocation_encoding(train_meta, y_train, other_meta, smoothing=TARGET_ENCODING_SMOOTHING):
    """Return cross-fitted train encodings and a full-train mapping for other rows."""
    train_meta = train_meta.reset_index(drop=True)
    other_meta = other_meta.reset_index(drop=True)
    y_train = np.asarray(y_train, dtype=float)

    def fit_mapping(meta, target):
        prior = float(np.mean(target))
        statistics = pd.DataFrame({
            "allocation": meta["ALLOCATION"].astype(str), "target": target
        }).groupby("allocation")["target"].agg(["count", "mean"])
        mapping = (statistics["count"] * statistics["mean"] + smoothing * prior) / (statistics["count"] + smoothing)
        return mapping, prior

    timestamps = pd.unique(train_meta["TS"].astype(str))
    train_encoded = np.full(len(train_meta), np.nan)
    inner_cv = KFold(n_splits=min(5, len(timestamps)), shuffle=True, random_state=RANDOM_STATE)
    for inner_train_pos, inner_valid_pos in inner_cv.split(timestamps):
        inner_train_mask = train_meta["TS"].astype(str).isin(timestamps[inner_train_pos]).to_numpy()
        inner_valid_mask = train_meta["TS"].astype(str).isin(timestamps[inner_valid_pos]).to_numpy()
        mapping, prior = fit_mapping(train_meta.loc[inner_train_mask], y_train[inner_train_mask])
        train_encoded[inner_valid_mask] = (
            train_meta.loc[inner_valid_mask, "ALLOCATION"].astype(str).map(mapping).fillna(prior)
        )

    full_mapping, full_prior = fit_mapping(train_meta, y_train)
    other_encoded = other_meta["ALLOCATION"].astype(str).map(full_mapping).fillna(full_prior).to_numpy()
    assert np.isfinite(train_encoded).all() and np.isfinite(other_encoded).all()
    return train_encoded, other_encoded, full_mapping, full_prior

def prepare_fold(train_idx, valid_idx):
    """Fit preprocessing on one outer training fold and transform its validation fold."""
    train_numeric = features_train.iloc[train_idx]
    valid_numeric = features_train.iloc[valid_idx]
    medians = train_numeric.median().fillna(0.0)
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric.fillna(medians))
    valid_scaled = scaler.transform(valid_numeric.fillna(medians))
    train_te, valid_te, _, _ = allocation_encoding(
        X_train.iloc[train_idx][["TS", "ALLOCATION"]], y_binary[train_idx],
        X_train.iloc[valid_idx][["TS", "ALLOCATION"]],
    )
    return np.column_stack([train_scaled, train_te]), np.column_stack([valid_scaled, valid_te])

## 7. Main feature ablations

**What:** summarize the controlled feature experiments saved in the executed project notebook.  
**Why:** these runs already isolate the useful hypotheses; rerunning all eleven would add time and clutter without changing the story.  
**Observation:** target encoding produced the largest gain, and cross-sectional plus confidence features were complementary. Return-volume interactions, timestamp-regime features, and the separate turnover block hurt accuracy.  
**Decision:** keep only the E10 representation. Values below are historical saved outputs, not newly estimated in this run.

In [8]:
ablation_results = pd.DataFrame([
    ("E0", "Benchmark features", 0.522194, "Reference"),
    ("E1", "+ cross-sectional context", 0.522297, "Useful in combination"),
    ("E2", "+ RET_1 confidence", 0.522348, "Useful in combination"),
    ("E3", "+ return-volume interactions", 0.522000, "Rejected"),
    ("E4", "+ timestamp-regime block", 0.521639, "Rejected"),
    ("E5", "+ turnover block", 0.521831, "Rejected"),
    ("E6", "+ cross-sectional + confidence", 0.522845, "Retained combination"),
    ("E7", "Allocation frequency encoding", 0.522196, "Neutral; rejected"),
    ("E8", "Allocation target encoding", 0.523570, "Retained"),
    ("E9", "Cross-sectional + target encoding", 0.523806, "Retained combination"),
    ("E10", "Cross-sectional + confidence + target encoding", 0.524658, "Selected representation"),
], columns=["experiment", "change", "mean_accuracy", "outcome"])
ablation_results["gain_vs_E0_pp"] = 100 * (ablation_results["mean_accuracy"] - ablation_results.loc[0, "mean_accuracy"])
display(ablation_results.round({"mean_accuracy": 6, "gain_vs_E0_pp": 4}))

,experiment,change,mean_accuracy,outcome,gain_vs_E0_pp
0,E0,Benchmark features,0.522194,Reference,0.0000
1,E1,+ cross-sectional context,0.522297,Useful in combination,0.0103
2,E2,+ RET_1 confidence,0.522348,Useful in combination,0.0154
3,E3,+ return-volume interactions,0.522000,Rejected,-0.0194
4,E4,+ timestamp-regime block,0.521639,Rejected,-0.0555
5,E5,+ turnover block,0.521831,Rejected,-0.0363
6,E6,+ cross-sectional + confidence,0.522845,Retained combination,0.0651
7,E7,Allocation frequency encoding,0.522196,Neutral; rejected,0.0002
8,E8,Allocation target encoding,0.523570,Retained,0.1376
9,E9,Cross-sectional + target encoding,0.523806,Retained combination,0.1612


## 8. Model comparison: LightGBM, XGBoost, and CatBoost

**What:** compare regression and classification objectives with the same E10 representation.  
**Why:** the leaderboard scores a sign, so direct classification may fit the objective better than predicting return magnitude.  
**Observation:** binary objectives beat regression for all three boosting libraries; CatBoost binary was best in the shuffled timestamp comparison.  
**Decision:** carry the three classifiers into purged validation. The table preserves the original executed eight-fold results.

In [9]:
historical_model_comparison = pd.DataFrame([
    ("Ridge", "regression", 0.521400, 0.521366, 0.516725),
    ("Logistic regression", "classification", 0.523493, 0.523453, 0.516799),
    ("LightGBM", "regression", 0.524080, 0.524075, 0.520938),
    ("LightGBM", "classification", 0.526040, 0.526018, 0.519188),
    ("XGBoost", "regression", 0.523454, 0.523444, 0.519425),
    ("XGBoost", "classification", 0.524714, 0.524694, 0.518787),
    ("CatBoost", "regression", 0.524115, 0.524102, 0.518832),
    ("CatBoost", "classification", 0.526312, 0.526293, 0.521131),
], columns=["model", "objective", "mean_accuracy", "global_oof_accuracy", "worst_fold_accuracy"])
display(historical_model_comparison.round(6))

,model,objective,mean_accuracy,global_oof_accuracy,worst_fold_accuracy
0,Ridge,regression,0.521400,0.521366,0.516725
1,Logistic regression,classification,0.523493,0.523453,0.516799
2,LightGBM,regression,0.524080,0.524075,0.520938
3,LightGBM,classification,0.526040,0.526018,0.519188
4,XGBoost,regression,0.523454,0.523444,0.519425
5,XGBoost,classification,0.524714,0.524694,0.518787
6,CatBoost,regression,0.524115,0.524102,0.518832
7,CatBoost,classification,0.526312,0.526293,0.521131


### Purged comparison of the three classifiers

The following cell recomputes the decisive comparison. Each model sees identical preprocessed folds and uses the original parameters:

- **LightGBM:** 500 trees, depth 3, 15 leaves, learning rate 0.02.
- **XGBoost:** 400 trees, depth 4, learning rate 0.03, row/column subsampling 0.9.
- **CatBoost:** 500 iterations, depth 5, learning rate 0.03, log loss.

Reusing each prepared fold across the three estimators makes the code faster without changing the methodology.

In [10]:
def make_classifier(name):
    if name == "LightGBM":
        return lgb.LGBMClassifier(
            objective="binary", n_estimators=500, learning_rate=0.02, max_depth=3,
            num_leaves=15, n_jobs=N_JOBS, random_state=RANDOM_STATE, verbosity=-1,
        )
    if name == "XGBoost":
        return XGBClassifier(
            n_estimators=400, learning_rate=0.03, max_depth=4, subsample=0.9,
            colsample_bytree=0.9, objective="binary:logistic", eval_metric="logloss",
            n_jobs=N_JOBS, random_state=RANDOM_STATE, verbosity=0,
        )
    if name == "CatBoost":
        return CatBoostClassifier(
            iterations=500, depth=5, learning_rate=0.03, loss_function="Logloss",
            verbose=False, random_seed=RANDOM_STATE, thread_count=N_JOBS, allow_writing_files=False,
        )
    raise KeyError(name)

def evaluate_classifiers(folds):
    """Evaluate all three classifiers on the same purged folds."""
    names = ["LightGBM", "XGBoost", "CatBoost"]
    oof = {name: np.full(len(X_train), np.nan) for name in names}
    fold_ids = np.full(len(X_train), -1, dtype=int)
    records = []

    for fold in folds:
        train_idx, valid_idx = fold["train_idx"], fold["valid_idx"]
        X_fold_train, X_fold_valid = prepare_fold(train_idx, valid_idx)
        fold_ids[valid_idx] = fold["fold"]
        for name in names:
            model_started = time.perf_counter()
            model = make_classifier(name)
            model.fit(X_fold_train, y_binary[train_idx])
            probability = model.predict_proba(X_fold_valid)[:, 1]
            oof[name][valid_idx] = probability
            records.append({
                "model": name, "fold": fold["fold"],
                "accuracy": accuracy_score(y_binary[valid_idx], probability >= 0.5),
                "validation_rows": len(valid_idx), "fit_seconds": time.perf_counter() - model_started,
            })
            del model
        del X_fold_train, X_fold_valid
        _ = gc.collect()

    fold_results = pd.DataFrame(records)
    summary = []
    covered = fold_ids >= 0
    for name in names:
        model_folds = fold_results.loc[fold_results["model"] == name, "accuracy"]
        summary.append({
            "model": name, "mean_accuracy": model_folds.mean(),
            "global_oof_accuracy": accuracy_score(y_binary[covered], oof[name][covered] >= 0.5),
            "worst_fold_accuracy": model_folds.min(), "best_fold_accuracy": model_folds.max(),
            "validated_rows": int(covered.sum()),
        })
    return oof, fold_ids, fold_results, pd.DataFrame(summary)

purged_oof, purged_fold_ids, purged_fold_results, purged_summary = evaluate_classifiers(purged_folds)
display(
    purged_summary.round(6),
    purged_fold_results.pivot(index="fold", columns="model", values="accuracy").round(6),
)

,model,mean_accuracy,global_oof_accuracy,worst_fold_accuracy,best_fold_accuracy,validated_rows
0,LightGBM,0.523603,0.523560,0.518104,0.529114,210757
1,XGBoost,0.523300,0.523257,0.518049,0.528187,210757
2,CatBoost,0.524127,0.524087,0.519499,0.530894,210757


model,CatBoost,LightGBM,XGBoost
fold,,,
0,0.519499,0.518104,0.518049
1,0.525031,0.523520,0.523480
2,0.521084,0.523674,0.523484
3,0.530894,0.529114,0.528187


### Model interpretation

- **LightGBM** establishes that the engineered representation remains useful under future-facing validation.
- **XGBoost** is competitive but does not improve on LightGBM or CatBoost.
- **CatBoost** is narrowly best in both the historical model comparison and the purged comparison.

The margins are small, so the choice is based on consistent validation evidence rather than a claim that one library is universally superior. With the current pinned runtime, CatBoost reaches 0.524087 pooled purged OOF accuracy versus 0.523741 in the legacy saved output; the ranking is unchanged, and the difference is reported as a library-version rerun rather than a model improvement.

## 9. Threshold calibration

**What:** tune the probability cutoff using only purged out-of-fold CatBoost predictions.  
**Why:** accuracy depends on the decision threshold, and a slightly shifted class prior made 0.5 suboptimal in the original work.  
**Observation:** the submitted project records a final threshold of 0.492; the current library stack re-estimates the all-OOF optimum at 0.495.  
**Decision:** keep 0.492 as the historical production choice. Use nested fold-by-fold calibration as the validation estimate, and show 0.495 only as a reproducibility diagnostic.

In [11]:
def best_threshold(target, probability, thresholds):
    """Select the best grid threshold efficiently; ties use their median."""
    order = np.argsort(probability, kind="stable")
    ordered_probability = probability[order]
    ordered_target = np.asarray(target, dtype=int)[order]
    cumulative_positive = np.concatenate([[0], np.cumsum(ordered_target, dtype=int)])
    positions = np.arange(len(ordered_target) + 1)
    cumulative_negative = positions - cumulative_positive
    split_positions = np.searchsorted(ordered_probability, thresholds, side="left")
    correct = cumulative_negative[split_positions] + cumulative_positive[-1] - cumulative_positive[split_positions]
    accuracies = correct / len(ordered_target)
    best_accuracy = accuracies.max()
    selected = float(np.median(thresholds[np.isclose(accuracies, best_accuracy)]))
    return selected, float(best_accuracy)

def calibrate_threshold(target, probability, fold_ids):
    thresholds = np.linspace(0.40, 0.60, 201)
    covered = np.isfinite(probability) & (fold_ids >= 0)
    nested_prediction = np.full(len(target), -1, dtype=int)
    records = []
    for fold in np.unique(fold_ids[covered]):
        selection = covered & (fold_ids != fold)
        validation = covered & (fold_ids == fold)
        threshold, _ = best_threshold(target[selection], probability[selection], thresholds)
        nested_prediction[validation] = probability[validation] >= threshold
        records.append({
            "fold": int(fold), "selected_threshold": threshold,
            "calibrated_accuracy": accuracy_score(target[validation], nested_prediction[validation]),
            "accuracy_at_0.5": accuracy_score(target[validation], probability[validation] >= 0.5),
            "validation_rows": int(validation.sum()),
        })
    rerun_threshold, apparent_accuracy = best_threshold(target[covered], probability[covered], thresholds)
    return {
        "rerun_global_threshold": rerun_threshold,
        "recorded_production_threshold": 0.492,
        "default_oof_accuracy": accuracy_score(target[covered], probability[covered] >= 0.5),
        "recorded_threshold_oof_accuracy": accuracy_score(target[covered], probability[covered] >= 0.492),
        "nested_calibrated_accuracy": accuracy_score(target[covered], nested_prediction[covered]),
        "apparent_accuracy_at_rerun_threshold": apparent_accuracy,
        "fold_results": pd.DataFrame(records),
        "validated_rows": int(covered.sum()),
    }

threshold_result = calibrate_threshold(y_binary, purged_oof["CatBoost"], purged_fold_ids)
FINAL_THRESHOLD = threshold_result["recorded_production_threshold"]
threshold_summary = pd.DataFrame([{key: value for key, value in threshold_result.items() if key != "fold_results"}])
display(threshold_result["fold_results"].round(6), threshold_summary.round(6))

,fold,selected_threshold,calibrated_accuracy,accuracy_at_0.5,validation_rows
0,0,0.495,0.518673,0.519499,54491
1,1,0.495,0.527387,0.525031,50936
2,2,0.495,0.523179,0.521084,52504
3,3,0.495,0.533961,0.530894,52826


,rerun_global_threshold,recorded_production_threshold,default_oof_accuracy,recorded_threshold_oof_accuracy,nested_calibrated_accuracy,apparent_accuracy_at_rerun_threshold,validated_rows
0,0.495,0.492,0.524087,0.525302,0.525733,0.525733,210757


The nested calibrated accuracy is the validation estimate. The 0.495 current-stack optimum is fitted and evaluated on the same pooled OOF rows, so it is shown only as a diagnostic. The final submission keeps the historically recorded 0.492 threshold.

## 10. Ensemble experiment

**What:** test non-negative blends of the three purged OOF probability vectors, using 0.05 weight increments and the same threshold grid.  
**Why:** an ensemble is useful only if different models make complementary errors.  
**Observation:** the original research rejected the blend because it did not improve nested validation.  
**Decision:** rerun the nested comparison and retain CatBoost unless the ensemble has a positive out-of-fold gain.

In [12]:
def evaluate_ensemble(target, oof_predictions, fold_ids, catboost_threshold, step=0.05):
    names = ["LightGBM", "CatBoost", "XGBoost"]
    score_matrix = np.column_stack([oof_predictions[name] for name in names])
    covered = np.isfinite(score_matrix).all(axis=1) & (fold_ids >= 0)
    thresholds = np.linspace(0.40, 0.60, 201)
    units = int(round(1 / step))
    weight_grid = [
        np.array(weights, dtype=float) / units
        for weights in itertools.product(range(units + 1), repeat=3)
        if sum(weights) == units
    ]

    def select(mask):
        selected_y, selected_scores = target[mask], score_matrix[mask]
        best = (-np.inf, None, None)
        for weights in weight_grid:
            blended = selected_scores @ weights
            threshold, accuracy = best_threshold(selected_y, blended, thresholds)
            if accuracy > best[0]:
                best = (accuracy, weights, threshold)
        return best

    ensemble_prediction = np.full(len(target), -1, dtype=int)
    catboost_prediction = np.full(len(target), -1, dtype=int)
    records = []
    for fold in np.unique(fold_ids[covered]):
        selection_mask = covered & (fold_ids != fold)
        validation_mask = covered & (fold_ids == fold)
        selection_accuracy, weights, threshold = select(selection_mask)
        ensemble_prediction[validation_mask] = score_matrix[validation_mask] @ weights >= threshold
        catboost_prediction[validation_mask] = oof_predictions["CatBoost"][validation_mask] >= catboost_threshold
        records.append({
            "fold": int(fold), "LightGBM_weight": weights[0], "CatBoost_weight": weights[1],
            "XGBoost_weight": weights[2], "threshold": threshold,
            "ensemble_accuracy": accuracy_score(target[validation_mask], ensemble_prediction[validation_mask]),
            "CatBoost_accuracy": accuracy_score(target[validation_mask], catboost_prediction[validation_mask]),
        })

    ensemble_accuracy = accuracy_score(target[covered], ensemble_prediction[covered])
    catboost_accuracy = accuracy_score(target[covered], catboost_prediction[covered])
    _, production_weights, production_threshold = select(covered)
    return {
        "fold_results": pd.DataFrame(records), "nested_ensemble_accuracy": ensemble_accuracy,
        "CatBoost_reference_accuracy": catboost_accuracy,
        "gain_vs_CatBoost": ensemble_accuracy - catboost_accuracy,
        "production_weights": dict(zip(names, production_weights)),
        "production_threshold": production_threshold,
        "retain_ensemble": ensemble_accuracy > catboost_accuracy,
    }

ensemble_result = evaluate_ensemble(y_binary, purged_oof, purged_fold_ids, FINAL_THRESHOLD)
ensemble_summary = pd.DataFrame([{key: value for key, value in ensemble_result.items() if key != "fold_results"}])
display(ensemble_result["fold_results"].round(6), ensemble_summary.round(6))

,fold,LightGBM_weight,CatBoost_weight,XGBoost_weight,threshold,ensemble_accuracy,CatBoost_accuracy
0,0,0.00,0.80,0.20,0.489,0.518159,0.518544
1,1,0.25,0.60,0.15,0.491,0.525856,0.526288
2,2,0.00,1.00,0.00,0.495,0.523179,0.524055
3,3,0.75,0.25,0.00,0.503,0.529114,0.532560


,nested_ensemble_accuracy,CatBoost_reference_accuracy,gain_vs_CatBoost,production_weights,production_threshold,retain_ensemble
0,0.524016,0.525302,-0.001286,"{'LightGBM': 0.25, 'CatBoost': 0.6, 'XGBoost':...",0.491,False


## 11. Central comparison table

This table separates historical shuffled-CV evidence from the rerun purged and nested results. It is the shortest complete account of how the final choice was made.

In [13]:
comparison_rows = [
    {"experiment": "RET_1 rule", "validation": "Deterministic / all train rows",
     "features": "RET_1 only", "accuracy": ret1_accuracy, "threshold": 0.0, "notes": "Simple reference"},
    {"experiment": "Official LightGBM benchmark", "validation": "8-fold shuffled timestamp CV",
     "features": "53 benchmark", "accuracy": official_benchmark["mean_accuracy"], "threshold": 0.0, "notes": "Regression sign"},
    {"experiment": "E10 feature ablation", "validation": "8-fold shuffled timestamp CV (saved)",
     "features": "Final 93-feature representation", "accuracy": 0.524658, "threshold": 0.0, "notes": "LightGBM regression"},
    {"experiment": "CatBoost binary", "validation": "8-fold shuffled timestamp CV (saved)",
     "features": "Final 93-feature representation", "accuracy": 0.526312, "threshold": 0.5, "notes": "Best shuffled-CV classifier"},
]
for row in purged_summary.to_dict(orient="records"):
    comparison_rows.append({
        "experiment": f"{row['model']} binary", "validation": "4-fold purged expanding CV",
        "features": "Final 93-feature representation", "accuracy": row["global_oof_accuracy"],
        "threshold": 0.5, "notes": "Pooled purged OOF",
    })
comparison_rows.extend([
    {"experiment": "CatBoost + nested calibration", "validation": "Nested purged OOF (current rerun)",
     "features": "Final 93-feature representation", "accuracy": threshold_result["nested_calibrated_accuracy"],
     "threshold": threshold_result["rerun_global_threshold"], "notes": "Fold thresholds are selected out of fold"},
    {"experiment": "Final recorded CatBoost pipeline", "validation": "Purged OOF at recorded threshold",
     "features": "Final 93-feature representation", "accuracy": threshold_result["recorded_threshold_oof_accuracy"],
     "threshold": FINAL_THRESHOLD, "notes": "Historical production threshold"},
    {"experiment": "Three-model ensemble", "validation": "Nested purged OOF",
     "features": "LightGBM + XGBoost + CatBoost", "accuracy": ensemble_result["nested_ensemble_accuracy"],
     "threshold": ensemble_result["production_threshold"],
     "notes": "Retained" if ensemble_result["retain_ensemble"] else "Rejected: no nested gain"},
])
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.round({"accuracy": 6, "threshold": 3}))

,experiment,validation,features,accuracy,threshold,notes
0,RET_1 rule,Deterministic / all train rows,RET_1 only,0.518924,0.000,Simple reference
1,Official LightGBM benchmark,8-fold shuffled timestamp CV,53 benchmark,0.522194,0.000,Regression sign
2,E10 feature ablation,8-fold shuffled timestamp CV (saved),Final 93-feature representation,0.524658,0.000,LightGBM regression
3,CatBoost binary,8-fold shuffled timestamp CV (saved),Final 93-feature representation,0.526312,0.500,Best shuffled-CV classifier
4,LightGBM binary,4-fold purged expanding CV,Final 93-feature representation,0.523560,0.500,Pooled purged OOF
5,XGBoost binary,4-fold purged expanding CV,Final 93-feature representation,0.523257,0.500,Pooled purged OOF
6,CatBoost binary,4-fold purged expanding CV,Final 93-feature representation,0.524087,0.500,Pooled purged OOF
7,CatBoost + nested calibration,Nested purged OOF (current rerun),Final 93-feature representation,0.525733,0.495,Fold thresholds are selected out of fold
8,Final recorded CatBoost pipeline,Purged OOF at recorded threshold,Final 93-feature representation,0.525302,0.492,Historical production threshold
9,Three-model ensemble,Nested purged OOF,LightGBM + XGBoost + CatBoost,0.524016,0.491,Rejected: no nested gain


## 12. Final model selection

**What:** freeze one production pipeline before touching the test set.  
**Why:** test predictions and the public leaderboard must not become model-selection tools.  
**Observation:** CatBoost was best under both model-comparison schemes, while the ensemble did not provide a robust nested gain.  
**Decision:** train one CatBoost binary classifier with the final 93-feature representation and threshold 0.492.

In [14]:
FINAL_MODEL = {
    "model": "CatBoostClassifier",
    "features": "53 benchmark + 24 cross-sectional + 15 RET_1 confidence + allocation target encoding",
    "preprocessing": "training median imputation + StandardScaler",
    "validation": "four purged expanding timestamp folds; 20-timestamp embargo",
    "threshold": FINAL_THRESHOLD,
    "ensemble": False,
}
display(pd.Series(FINAL_MODEL, name="final pipeline").to_frame())

,final pipeline
model,CatBoostClassifier
features,53 benchmark + 24 cross-sectional + 15 RET_1 c...
preprocessing,training median imputation + StandardScaler
validation,four purged expanding timestamp folds; 20-time...
threshold,0.492
ensemble,False


## 13. Final training

**What:** fit the selected pipeline on all labelled rows.  
**Why:** once model, features, and threshold are fixed, all available training data can be used.  
**Decision:** fit preprocessing and allocation statistics on the full training set, then train a fresh CatBoost model with the validated parameters.

In [15]:
def train_final_model(train_features, train_meta, target, test_features, test_meta):
    """Fit final preprocessing and CatBoost; return the fitted bundle and transformed test data."""
    medians = train_features.median().fillna(0.0)
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_features.fillna(medians))
    test_scaled = scaler.transform(test_features.fillna(medians))
    train_te, test_te, mapping, prior = allocation_encoding(
        train_meta[["TS", "ALLOCATION"]], target, test_meta[["TS", "ALLOCATION"]]
    )
    train_matrix = np.column_stack([train_scaled, train_te])
    test_matrix = np.column_stack([test_scaled, test_te])
    model = make_classifier("CatBoost")
    started = time.perf_counter()
    model.fit(train_matrix, target)
    bundle = {
        "model": model, "medians": medians, "scaler": scaler,
        "allocation_mapping": mapping, "allocation_prior": prior,
        "feature_names": list(train_features.columns) + ["ALLOCATION__TARGET_ENCODED"],
        "training_seconds": time.perf_counter() - started,
    }
    return bundle, test_matrix

features_test = build_features(X_test)
assert list(features_test.columns) == list(features_train.columns)
final_bundle, final_test_matrix = train_final_model(
    features_train, X_train, y_binary, features_test, X_test
)
print({
    "training_rows": len(X_train),
    "model_features": len(final_bundle["feature_names"]),
    "training_seconds": round(final_bundle["training_seconds"], 1),
})

{'training_rows': 527073, 'model_features': 93, 'training_seconds': 85.8}


## 14. Test prediction and submission

**What:** convert CatBoost probabilities to binary predictions and write the required two-column CSV.  
**Why:** submission order and schema are part of reproducibility.  
**Decision:** align predictions by `ROW_ID`, validate every field, and save the generated file under the ignored `submissions/` directory.

In [16]:
test_probability = final_bundle["model"].predict_proba(final_test_matrix)[:, 1]
test_prediction = (test_probability >= FINAL_THRESHOLD).astype(int)

prediction_by_id = pd.Series(test_prediction, index=X_test["ROW_ID"].to_numpy())
submission = sample_submission[["ROW_ID"]].copy()
submission["prediction"] = submission["ROW_ID"].map(prediction_by_id)

assert len(submission) == len(sample_submission)
assert submission["ROW_ID"].equals(sample_submission["ROW_ID"])
assert submission["prediction"].notna().all()
assert set(submission["prediction"].unique()).issubset({0, 1})
submission["prediction"] = submission["prediction"].astype(int)

submission_dir = PROJECT_DIR / "submissions"
submission_dir.mkdir(exist_ok=True)
submission_path = submission_dir / "submission_final_verified.csv"
submission.to_csv(submission_path, index=False)

submission_check = pd.read_csv(submission_path)
assert submission_check.equals(submission)
print({
    "file": submission_path.relative_to(PROJECT_DIR).as_posix(),
    "rows": len(submission),
    "predicted_positive_rate": round(float(submission["prediction"].mean()), 6),
    "schema_valid": True,
})

{'file': 'submissions/submission_final_verified.csv', 'rows': 31870, 'predicted_positive_rate': 0.677565, 'schema_valid': True}


## 15. Results and Key Takeaways

- **Problem:** predict the sign of a future allocation return; accuracy is the metric.
- **Validation:** timestamp-level folds were essential, and purged expanding validation was the final model-selection view.
- **Best model:** CatBoost binary classification with the E10 feature representation and a 0.492 probability threshold.
- **What did not help:** return-volume interactions, timestamp-regime features, the standalone turnover block, allocation frequency, and the three-model ensemble.
- **Why this pipeline:** it was the strongest consistent individual model and remained simple enough to audit and reproduce.
- **Main lesson:** robust validation and controlled experiments mattered more than adding model complexity.

### Reported challenge result

- Purged cross-validation and nested threshold results are shown above and are **validation estimates**.
- **Public leaderboard accuracy: 0.5150.** This is the score achieved by the freshly reproduced submission on the official QRT platform. It is an external challenge result, not a validation metric, and it was not used to tune the pipeline.